# Text-Completion Language Models

The `llms.py` module defines the base interfaces and shared execution machinery for traditional text-completion language models.

It provides retry construction, synchronous and asynchronous prompt-cache helpers, the abstract `BaseLLM` Runnable interface, callback and LangSmith tracing, batching, streaming, generation, serialization, persistence, and the simpler `LLM` subclass interface built around a single-prompt `_call` method.

### Functions

1. `create_base_retry_decorator`: Creates a Tenacity retry decorator for selected exception types.

   Retries use exponential backoff beginning at four seconds and capped at ten seconds. The operation is attempted at most `max_retries` times, and the final exception is re-raised.

   Before each retry, the failure is logged and the supplied run manager receives an `on_retry` callback. Asynchronous callbacks are scheduled on the active event loop when one is running or executed through `asyncio.run` otherwise. Errors raised by an asynchronous retry callback are logged once and suppressed.

   At least one exception type must be supplied because the first entry initializes the retry predicate.

   * **Syntax:**
     ```python
     create_base_retry_decorator(
         error_types: list[
             type[BaseException]
         ], # Exception types that trigger a retry
         max_retries: int = 1, # Maximum number of attempts
         run_manager: AsyncCallbackManagerForLLMRun
         | CallbackManagerForLLMRun
         | None = None # Optional callback manager notified before retries
     ) -> Callable[
         [Any],
         Any
     ]
     ```

2. `get_prompts`: Looks up prompts in the configured synchronous cache.

   The cache key combines each prompt with a string created from the sorted model parameters. Cached generation lists are returned by their original prompt indexes, while cache misses are returned as both indexes and prompt strings.

   When caching is disabled or no global cache is configured for `cache=None`, every returned collection remains empty. A `ValueError` is raised when `cache=True` but no global cache has been configured.

   * **Syntax:**
     ```python
     get_prompts(
         params: dict[
             str,
             Any
         ], # Model and invocation parameters used in the cache key
         prompts: list[str], # Prompts to look up
         cache: BaseCache
         | bool
         | None = None # Explicit, global, or disabled cache selection
     ) -> tuple[
         dict[
             int,
             list[Generation]
         ],
         str,
         list[int],
         list[str]
     ]
     ```

3. `aget_prompts`: Asynchronously looks up prompts in the configured cache.

   It follows the same cache-key, hit, miss, and error behaviour as `get_prompts`, but performs lookups through `BaseCache.alookup`.

   * **Syntax:**
     ```python
     async aget_prompts(
         params: dict[
             str,
             Any
         ], # Model and invocation parameters used in the cache key
         prompts: list[str], # Prompts to look up
         cache: BaseCache
         | bool
         | None = None # Explicit, global, or disabled cache selection
     ) -> tuple[
         dict[
             int,
             list[Generation]
         ],
         str,
         list[int],
         list[str]
     ]
     ```

4. `update_cache`: Inserts newly generated results into the synchronous cache and the indexed result mapping.

   Each generated result is associated with the corresponding entry in `missing_prompt_idxs`. Cache updates are skipped when caching resolves to `None`.

   The provider-specific `llm_output` from `new_results` is returned. A `ValueError` is raised when `cache=True` but no global cache exists.

   * **Syntax:**
     ```python
     update_cache(
         cache: BaseCache
         | bool
         | None, # Explicit, global, or disabled cache selection
         existing_prompts: dict[
             int,
             list[Generation]
         ], # Indexed cached and newly generated results
         llm_string: str, # Model-and-parameter cache-key component
         missing_prompt_idxs: list[int], # Original indexes of generated prompts
         new_results: LLMResult, # Newly generated results
         prompts: list[str] # Complete original prompt list
     ) -> dict[
         str,
         Any
     ] | None
     ```

5. `aupdate_cache`: Asynchronously inserts newly generated results into the cache and indexed result mapping.

   It follows the same mapping and return behaviour as `update_cache`, but writes through `BaseCache.aupdate`.

   * **Syntax:**
     ```python
     async aupdate_cache(
         cache: BaseCache
         | bool
         | None, # Explicit, global, or disabled cache selection
         existing_prompts: dict[
             int,
             list[Generation]
         ], # Indexed cached and newly generated results
         llm_string: str, # Model-and-parameter cache-key component
         missing_prompt_idxs: list[int], # Original indexes of generated prompts
         new_results: LLMResult, # Newly generated results
         prompts: list[str] # Complete original prompt list
     ) -> dict[
         str,
         Any
     ] | None
     ```

# BaseLLM

`BaseLLM` is the abstract base class for text-completion language models.

It accepts strings, `PromptValue` objects, and message-like sequences. Normal invocation returns a string, streaming yields string chunks, and prompt-generation methods return `LLMResult` objects that may contain multiple candidate generations and provider-specific output.

Concrete subclasses must implement `_generate` and `_llm_type`. They may override `_agenerate`, `_stream`, `_astream`, and `_get_ls_params` to provide native asynchronous, streaming, or provider-specific tracing behaviour.

## Bases

- `BaseLanguageModel[str]`
- `ABC`

## Configuration

1. `model_config`: Allows arbitrary Python types in the Pydantic model.
   * **Definition:**
     ```python
     model_config = ConfigDict(
         arbitrary_types_allowed=True
     )
     ```

### Properties

1. `OutputType`: Returns the Runnable output type.
   * **Type:**
     ```python
     OutputType: type[str]
     ```

   * **Value:**
     ```python
     str
     ```

2. `_llm_type`: Returns the provider-specific LLM type used in identifying dictionaries.

   Concrete subclasses must implement this abstract property.

   * **Type:**
     ```python
     _llm_type: str
     ```

### Methods

1. `_get_ls_params`: Returns standardized LangSmith tracing parameters.

   The base provider name is derived by removing the `"LLM"` suffix from the concrete class name and converting it to lowercase. The model type is set to `"llm"`.

   Stop strings, model name, temperature, and maximum-token values are added when they can be found in invocation arguments or compatible model attributes.

   Provider integrations may override this protected method to supply stable provider-specific values.

   * **Syntax:**
     ```python
     _get_ls_params(
         self,
         stop: list[str] | None = None, # Stop substrings
         **kwargs: Any # Invocation-specific model parameters
     ) -> LangSmithParams
     ```

2. `invoke`: Performs one synchronous LLM invocation and returns the top generated text.

   The input is converted to a prompt value and sent through `generate_prompt`. Runnable configuration supplies callbacks, tags, metadata, run name, and an optional run ID.

   A `ValueError` is raised when the input is neither a `PromptValue`, string, nor message-like sequence.

   * **Syntax:**
     ```python
     invoke(
         self,
         input: LanguageModelInput, # Prompt value, string, or message-like sequence
         config: RunnableConfig | None = None, # Runnable configuration
         *,
         stop: list[str] | None = None, # Stop substrings
         **kwargs: Any # Provider-specific invocation parameters
     ) -> str
     ```

3. `ainvoke`: Performs one asynchronous LLM invocation and returns the top generated text.

   It follows the same input conversion and configuration handling as `invoke`, but delegates to `agenerate_prompt`.

   * **Syntax:**
     ```python
     async ainvoke(
         self,
         input: LanguageModelInput, # Prompt value, string, or message-like sequence
         config: RunnableConfig | None = None, # Runnable configuration
         *,
         stop: list[str] | None = None, # Stop substrings
         **kwargs: Any # Provider-specific invocation parameters
     ) -> str
     ```

4. `batch`: Processes multiple inputs synchronously.

   Without `max_concurrency`, all converted inputs are passed to one `generate_prompt` call so implementations can use provider batching. The first generation text for each input is returned.

   When `max_concurrency` is configured, inputs are divided into consecutive batches of that size and processed recursively with the concurrency limit cleared.

   If generation fails and `return_exceptions=True`, the same caught exception is returned for every input in the failing batch. Otherwise, the exception is re-raised. An empty input list returns an empty list.

   * **Syntax:**
     ```python
     batch(
         self,
         inputs: list[
             LanguageModelInput
         ], # Inputs to process
         config: RunnableConfig
         | list[RunnableConfig]
         | None = None, # Shared or per-input configuration
         *,
         return_exceptions: bool = False, # Return failures instead of raising them
         **kwargs: Any # Provider-specific generation parameters
     ) -> list[str]
     ```

5. `abatch`: Processes multiple inputs asynchronously.

   It follows the same batching, `max_concurrency`, empty-input, and exception-return behaviour as `batch`, while delegating to `agenerate_prompt`.

   * **Syntax:**
     ```python
     async abatch(
         self,
         inputs: list[
             LanguageModelInput
         ], # Inputs to process
         config: RunnableConfig
         | list[RunnableConfig]
         | None = None, # Shared or per-input configuration
         *,
         return_exceptions: bool = False, # Return failures instead of raising them
         **kwargs: Any # Provider-specific generation parameters
     ) -> list[str]
     ```

6. `stream`: Streams generated text synchronously.

   When `_stream` is not overridden, the complete result of `invoke` is yielded as one string.

   Native streaming creates an LLM trace, calls `_stream`, yields each `GenerationChunk.text`, and accumulates the chunks for the completion callback. Provider errors trigger `on_llm_error` with partial-generation data before being re-raised.

   A `ValueError` is raised and traced as an error when native streaming returns no chunks.

   * **Syntax:**
     ```python
     stream(
         self,
         input: LanguageModelInput, # Prompt value, string, or message-like sequence
         config: RunnableConfig | None = None, # Runnable configuration
         *,
         stop: list[str] | None = None, # Stop substrings
         **kwargs: Any # Provider-specific streaming parameters
     ) -> Iterator[str]
     ```

7. `astream`: Streams generated text asynchronously.

   When neither `_astream` nor `_stream` is overridden, the complete result of `ainvoke` is yielded once.

   Native asynchronous streaming creates an asynchronous trace, yields each chunk's text, accumulates partial output, reports failures through `on_llm_error`, and reports the combined generation through `on_llm_end`.

   A `ValueError` is raised when no generation chunks are produced.

   * **Syntax:**
     ```python
     async astream(
         self,
         input: LanguageModelInput, # Prompt value, string, or message-like sequence
         config: RunnableConfig | None = None, # Runnable configuration
         *,
         stop: list[str] | None = None, # Stop substrings
         **kwargs: Any # Provider-specific streaming parameters
     ) -> AsyncIterator[str]
     ```

8. `_generate`: Generates results synchronously for a list of prompt strings.

   This is the primary required implementation hook for concrete `BaseLLM` subclasses. Implementations may return multiple candidate `Generation` objects per prompt and provider-specific `llm_output`.

   * **Syntax:**
     ```python
     @abstractmethod
     _generate(
         self,
         prompts: list[str], # Prompts to process
         stop: list[str] | None = None, # Stop substrings
         run_manager: CallbackManagerForLLMRun | None = None, # Synchronous run manager
         **kwargs: Any # Provider-specific generation parameters
     ) -> LLMResult
     ```

9. `_agenerate`: Generates results asynchronously for a list of prompts.

   The base implementation runs `_generate` in an executor and converts an asynchronous run manager to its synchronous form. Providers may override this protected method with a native asynchronous API.

   * **Syntax:**
     ```python
     async _agenerate(
         self,
         prompts: list[str], # Prompts to process
         stop: list[str] | None = None, # Stop substrings
         run_manager: AsyncCallbackManagerForLLMRun | None = None, # Asynchronous run manager
         **kwargs: Any # Provider-specific generation parameters
     ) -> LLMResult
     ```

10. `_stream`: Streams `GenerationChunk` objects for one prompt.

    Providers may override this optional protected hook. The base implementation raises `NotImplementedError`, causing the public `stream` method to use non-streaming invocation as its fallback.

    * **Syntax:**
      ```python
      _stream(
          self,
          prompt: str, # Prompt to process
          stop: list[str] | None = None, # Stop substrings
          run_manager: CallbackManagerForLLMRun | None = None, # Synchronous run manager
          **kwargs: Any # Provider-specific streaming parameters
      ) -> Iterator[
          GenerationChunk
      ]
      ```

11. `_astream`: Asynchronously streams `GenerationChunk` objects.

    The base implementation creates the synchronous `_stream` iterator in an executor and advances that iterator through the executor until exhaustion. Providers may override it with a native asynchronous stream.

    * **Syntax:**
      ```python
      async _astream(
          self,
          prompt: str, # Prompt to process
          stop: list[str] | None = None, # Stop substrings
          run_manager: AsyncCallbackManagerForLLMRun | None = None, # Asynchronous run manager
          **kwargs: Any # Provider-specific streaming parameters
      ) -> AsyncIterator[
          GenerationChunk
      ]
      ```

12. `generate_prompt`: Converts prompt values to strings and delegates to `generate`.
    * **Syntax:**
      ```python
      generate_prompt(
          self,
          prompts: list[
              PromptValue
          ], # Prompt values to process
          stop: list[str] | None = None, # Stop substrings
          callbacks: Callbacks
          | list[Callbacks]
          | None = None, # Shared or per-prompt callbacks
          **kwargs: Any # Generation and tracing parameters
      ) -> LLMResult
      ```

13. `agenerate_prompt`: Converts prompt values to strings and delegates to `agenerate`.
    * **Syntax:**
      ```python
      async agenerate_prompt(
          self,
          prompts: list[
              PromptValue
          ], # Prompt values to process
          stop: list[str] | None = None, # Stop substrings
          callbacks: Callbacks
          | list[Callbacks]
          | None = None, # Shared or per-prompt callbacks
          **kwargs: Any # Generation and tracing parameters
      ) -> LLMResult
      ```

14. `generate`: Generates synchronous results for a sequence of prompt strings.

    The method configures one callback manager and trace per prompt, adds standard LangSmith parameters to metadata, checks the configured cache, generates only cache misses, updates the cache, restores results to original prompt order, and returns an `LLMResult`.

    A model with no active cache generates all prompts directly. When a cache is active, callback start events are created only for missing prompts.

    Per-prompt callback, tag, metadata, and run-name lists must match the prompt count. A single value may instead be shared by all prompts. Run IDs may be supplied as one UUID or a per-prompt list.

    A `ValueError` is raised when `prompts` is not a list or when per-prompt configuration lengths do not match.

    * **Syntax:**
      ```python
      generate(
          self,
          prompts: list[str], # Prompt strings to process
          stop: list[str] | None = None, # Stop substrings
          callbacks: Callbacks
          | list[Callbacks]
          | None = None, # Shared or per-prompt callbacks
          *,
          tags: list[str]
          | list[
              list[str]
          ]
          | None = None, # Shared or per-prompt tags
          metadata: dict[
              str,
              Any
          ]
          | list[
              dict[
                  str,
                  Any
              ]
          ]
          | None = None, # Shared or per-prompt metadata
          run_name: str
          | list[str]
          | None = None, # Shared or per-prompt run names
          run_id: UUID
          | list[
              UUID | None
          ]
          | None = None, # Shared or per-prompt run IDs
          **kwargs: Any # Provider-specific generation parameters
      ) -> LLMResult
      ```

15. `agenerate`: Asynchronously generates results for a sequence of prompt strings.

    It follows the same cache, ordering, configuration validation, tracing, and result-reconstruction behaviour as `generate`.

    Callback start operations are awaited concurrently. The provider-level asynchronous generation hook receives the generated prompt subset and an asynchronous run manager.

    * **Syntax:**
      ```python
      async agenerate(
          self,
          prompts: list[str], # Prompt strings to process
          stop: list[str] | None = None, # Stop substrings
          callbacks: Callbacks
          | list[Callbacks]
          | None = None, # Shared or per-prompt callbacks
          *,
          tags: list[str]
          | list[
              list[str]
          ]
          | None = None, # Shared or per-prompt tags
          metadata: dict[
              str,
              Any
          ]
          | list[
              dict[
                  str,
                  Any
              ]
          ]
          | None = None, # Shared or per-prompt metadata
          run_name: str
          | list[str]
          | None = None, # Shared or per-prompt run names
          run_id: UUID
          | list[
              UUID | None
          ]
          | None = None, # Shared or per-prompt run IDs
          **kwargs: Any # Provider-specific generation parameters
      ) -> LLMResult
      ```

16. `__str__`: Returns a display string containing the bold concrete class name and identifying parameters.
    * **Syntax:**
      ```python
      __str__(
          self
      ) -> str
      ```

17. `dict`: Returns the model's identifying dictionary.

    This method is deprecated in favour of `asdict` and is scheduled for removal in version `2.0.0`.

    * **Syntax:**
      ```python
      dict(
          self,
          **_kwargs: Any # Deprecated compatibility arguments
      ) -> dict[
          str,
          Any
      ]
      ```

18. `asdict`: Returns the model's identifying parameters together with its `_llm_type`.

    The model type is stored under the `"_type"` key.

    * **Syntax:**
      ```python
      asdict(
          self
      ) -> dict[
          str,
          Any
      ]
      ```

19. `save`: Saves the identifying dictionary as JSON or YAML.

    Missing parent directories are created automatically. Files ending in `.json` use indented JSON; `.yaml` and `.yml` use YAML.

    A `ValueError` is raised for any other file suffix. File-system and serialization errors are allowed to propagate.

    * **Syntax:**
      ```python
      save(
          self,
          file_path: Path | str # Destination JSON or YAML path
      ) -> None
      ```

## Cache Behaviour

Cache selection follows these rules:

- A `BaseCache` instance is used directly.
- `cache=True` requires a configured global cache.
- `cache=False` disables caching.
- `cache=None` uses the global cache when one exists and otherwise disables caching.
- Unsupported cache values raise `ValueError`.

The cache key is the prompt plus a string representation of sorted identifying and invocation parameters.

## Callback and Tracing Behaviour

Generation and streaming operations include:

- Serialized model information.
- Invocation parameters and stop options.
- Model-level and invocation-level callbacks.
- Tags and metadata.
- Filtered invocation parameters as inheritable LangSmith metadata.
- Provider, model, temperature, maximum-token, and stop metadata when available.
- Per-prompt run IDs.
- Completion callbacks containing flattened prompt results.
- Error callbacks when provider generation or streaming fails.

# LLM

`LLM` is a simpler abstract interface for text-completion models that generate one string from one prompt.

Concrete subclasses implement `_call`. The class supplies batched `_generate` and `_agenerate` wrappers and an executor-based asynchronous fallback.

## Bases

- `BaseLLM`

### Methods

1. `_call`: Generates one string synchronously from one prompt.

   This is the required implementation hook for concrete `LLM` subclasses. Returned text should not include the input prompt.

   When stop sequences are unsupported, an implementation may raise `NotImplementedError`.

   * **Syntax:**
     ```python
     @abstractmethod
     _call(
         self,
         prompt: str, # Prompt to process
         stop: list[str] | None = None, # Stop substrings
         run_manager: CallbackManagerForLLMRun | None = None, # Synchronous run manager
         **kwargs: Any # Provider-specific generation parameters
     ) -> str
     ```

2. `_acall`: Asynchronously generates one string from one prompt.

   The base implementation runs `_call` in an executor and converts an asynchronous run manager to its synchronous form. Providers may override it with a native asynchronous API.

   * **Syntax:**
     ```python
     async _acall(
         self,
         prompt: str, # Prompt to process
         stop: list[str] | None = None, # Stop substrings
         run_manager: AsyncCallbackManagerForLLMRun | None = None, # Asynchronous run manager
         **kwargs: Any # Provider-specific generation parameters
     ) -> str
     ```

3. `_generate`: Generates an `LLMResult` synchronously for multiple prompts.

   Prompts are processed sequentially through `_call`, and each returned string is wrapped in one `Generation`.

   For backward compatibility, the run manager is passed only when `_call` declares a `run_manager` parameter.

   * **Syntax:**
     ```python
     _generate(
         self,
         prompts: list[str], # Prompts to process
         stop: list[str] | None = None, # Stop substrings
         run_manager: CallbackManagerForLLMRun | None = None, # Synchronous run manager
         **kwargs: Any # Provider-specific generation parameters
     ) -> LLMResult
     ```

4. `_agenerate`: Generates an `LLMResult` asynchronously for multiple prompts.

   Prompts are processed sequentially through `_acall`, and each returned string is wrapped in one `Generation`.

   For backward compatibility, the asynchronous run manager is passed only when `_acall` declares a `run_manager` parameter.

   * **Syntax:**
     ```python
     async _agenerate(
         self,
         prompts: list[str], # Prompts to process
         stop: list[str] | None = None, # Stop substrings
         run_manager: AsyncCallbackManagerForLLMRun | None = None, # Asynchronous run manager
         **kwargs: Any # Provider-specific generation parameters
     ) -> LLMResult
     ```

## Internal Components Omitted

The following private implementation details are not documented as standalone API entries:

- `_log_error_once`
- `_resolve_cache`
- `_background_tasks`
- Input-conversion and cached-serialization helpers
- Generation callback helpers
- Run-ID normalization helper
- Deprecated-override compatibility helper
- Internal asynchronous single-prompt wrapper